# Week 7 - Outlier Detection and Data Quality
Calculate IQR boundaries and flag outliers (specifically add flags for ClosePrice, LivingArea, and DaysOnMarket)

In [1]:
import pandas as pd
from pathlib import Path

In [3]:
# load in dataset
data_dir = Path('../data/feature_engineered')

sold = pd.read_csv(
    data_dir / 'sold_feature_engineered.csv',
    low_memory=False
)

print('Sold Shape -', sold.shape) # this should output as (413853, 104)
sold.head()

Sold Shape - (413853, 104)


,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,close_to_original_list_ratio,Year,Month,YrMo,listing_to_contract_days,contract_to_close_days,elementary_school_district,high_school_district,unified_school_district,school_district_matched_flag
0,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,2024-01-26,240000.0,...,0.480962,2024,1,2024-01,777.0,65.0,San Mateo-Foster City,San Mateo Union High,NaN,True
1,NaN,False,NaN,NaN,False,759900.0,522107581,mdarwich12@gmail.com,2024-01-05,815000.0,...,1.072510,2024,1,2024-01,114.0,919.0,National Elementary,Sweetwater Union High,NaN,True
2,NaN,False,NaN,NaN,False,739900.0,510919001,mdarwich12@gmail.com,2024-01-05,810000.0,...,1.094743,2024,1,2024-01,255.0,778.0,National Elementary,Sweetwater Union High,NaN,True
3,NaN,True,NaN,NaN,False,2100000.0,1067652762,jenniferarielkennedy@gmail.com,2024-01-02,2100000.0,...,1.000000,2024,1,2024-01,0.0,48.0,NaN,NaN,San Luis Coastal Unified,True
4,"Tile,Wood",True,NaN,NaN,NaN,NaN,1061988701,dannyduong@gmail.com,2024-01-31,2340000.0,...,NaN,2024,1,2024-01,33.0,-33.0,Evergreen Elementary,East Side Union High,NaN,True


In [5]:
# cehck for any missing values in the key numeric fields we need to apply IQR filtering to
# DO NOT delete records outright
numeric_columns = [
    'ClosePrice',
    'LivingArea',
    'DaysOnMarket'
]

# convert columns to numeric if needed
for column in numeric_columns:
    sold[column] = pd.to_numeric(sold[column], errors='coerce')

display(sold[numeric_columns].describe())

print('Missing Values:')
print(sold[numeric_columns].isna().sum())

,ClosePrice,LivingArea,DaysOnMarket
count,4.138530e+05,4.136190e+05,413853.000000
mean,1.193067e+06,1.904507e+03,37.357994
std,6.290538e+06,2.648261e+04,53.580019
min,1.150000e+00,1.000000e+00,0.000000
25%,5.750000e+05,1.248000e+03,8.000000
50%,8.230000e+05,1.643000e+03,18.000000
75%,1.300000e+06,2.219000e+03,48.000000
max,9.895000e+08,1.702132e+07,12430.000000


Missing Values:
ClosePrice        0
LivingArea      234
DaysOnMarket      0
dtype: int64


### 1) Implement Interquartile Range (IQR) method to identify and remove these records

The IQR method removes records that fall outside a defined statistical range which is given as: $Q3 - Q1$
$$\text{Lower} = Q1 - 1.5(IQR)$$
$$\text{Upper} = Q3 + 1.5(IQR)$$

There's the `quantile()` function from panads where we can call the method on a column to calculate the quantiles.

In [ ]:
# calculate IQR boundaries for each numeric field
iqr_summary = []
for column in numeric_columns:
    q1 = sold[column].quantile(0.25)
    q3 = sold[column].quantile(0.75)

    iqr = q3 - q1
    lower = q1 - (1.5 * iqr)
    upper = q3 + (1.5 * iqr)

    iqr_summary.append({
        'Column' : column,
        'Q1': q1,
        'Q3': q3,
        'IQR' : iqr,
        'LowerBound' : lower,
        'UpperBound' : upper
    })

iqr_summary = pd.DataFrame(iqr_summary)

iqr_summary

,Column,Q1,Q3,IQR,LowerBound,UpperBound
0,ClosePrice,575000.0,1300000.0,725000.0,-512500.0,2387500.0
1,LivingArea,1248.0,2219.0,971.0,-208.5,3675.5
2,DaysOnMarket,8.0,48.0,40.0,-52.0,108.0


### 2) Flag outliers
Flag the outliers from the columns: (ClosePrice, LivingArea, DaysOnMarket)